# Live AgentPinBoard demo with WebSocket streaming

This notebook starts a WebSocket server that pushes graph deltas as the agent
populates the graph, and a browser viewer renders them live with Cytoscape.js.

Open `http://localhost:8765/` in a browser **after** running the server cell.
The page is served by the same process; the WS upgrade also lives on `/`.

Uses a deterministic `MockChatModel` so it works without an LLM provider —
swap it for any LangChain `BaseChatModel` for a real agent.

## 0. Imports + setup

In [ ]:
from __future__ import annotations

import asyncio
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langgraph.prebuilt import ToolRuntime
from langgraph.store.memory import InMemoryStore
from pydantic import BaseModel, Field

from agent_pinboard import Entity, make_graph_tools, node, pin
from agent_pinboard.integrations.websocket_hook import (
    WebSocketHook,
    serve_websocket,
)


## 1. Entities + models + tools (same as the basic demo)

In [ ]:
import ipaddress

def canonical_ip(v: str) -> str:
    return str(ipaddress.ip_address(v).compressed)

IP = Entity(name="IP", description="ipv4/ipv6", normalizer=canonical_ip)
User = Entity(name="User", description="actor", normalizer=lambda v: str(v).strip())
Action = Entity(name="Action", description="API action")

class Actor(BaseModel):
    user_arn: str | None = node(type=User, description="who", default=None)

class CloudTrailEvent(BaseModel):
    src_ip: str | None = node(type=IP, description="src ip", default=None)
    actor: Actor | None = None
    action_name: str | None = node(type=Action, description="action", default=None)
    event_time: datetime | None = Field(default=None)

class VTReport(BaseModel):
    queried: str = node(type=IP, description="queried IP")
    related_ips: list[str] = node(type=IP, description="related", default_factory=list)
    score: int = Field(default=0)

@pin(model=CloudTrailEvent, many=True)
@tool
def fetch_cloudtrail(user_arn: str, runtime: ToolRuntime) -> list[dict]:
    """Mock CloudTrail fetch."""
    return [
        {"src_ip": "185.220.101.42", "actor": {"user_arn": user_arn},
         "action_name": "AssumeRole", "event_time": datetime.now(UTC).isoformat()},
        {"src_ip": "185.220.101.42", "actor": {"user_arn": user_arn},
         "action_name": "ListBuckets", "event_time": datetime.now(UTC).isoformat()},
    ]

@pin(model=VTReport)
@tool
def vt_lookup(value: str, runtime: ToolRuntime) -> dict:
    """Mock VirusTotal lookup."""
    return {"queried": value, "related_ips": ["45.77.0.1", "8.8.8.8"], "score": 87}


## 2. Mock LLM and a deterministic plan

(see `agent_demo.ipynb` for the full explanation)

In [ ]:
class MockChatModel(BaseChatModel):
    plan: list[dict[str, Any]] = []
    def __init__(self, plan: list[dict[str, Any]]) -> None:
        super().__init__()
        object.__setattr__(self, "plan", plan)
    @property
    def _llm_type(self) -> str: return "mock"
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        cursor = sum(1 for m in messages if isinstance(m, ToolMessage))
        if cursor < len(self.plan):
            step = self.plan[cursor]
            ai = AIMessage(content="", tool_calls=[{
                "name": step["tool"], "args": step["args"],
                "id": f"call-{cursor}", "type": "tool_call",
            }])
        else:
            ai = AIMessage(content="Investigation complete.")
        return ChatResult(generations=[ChatGeneration(message=ai)])
    def bind_tools(self, tools, **_kwargs): return self

PLAN = [
    {"tool": "graph_summary", "args": {}},
    {"tool": "fetch_cloudtrail", "args": {"user_arn": "arn:aws:iam::123:user/admin"}},
    {"tool": "vt_lookup", "args": {"value": "185.220.101.42"}},
    {"tool": "explore", "args": {"node_type": "IP", "value": "185.220.101.42"}},
    {"tool": "find_path", "args": {
        "from_type": "IP", "from_value": "185.220.101.42",
        "to_type": "IP", "to_value": "8.8.8.8",
    }},
    {"tool": "timeline", "args": {
        "node_type": "User", "value": "arn:aws:iam::123:user/admin",
    }},
]


## 3. Start the WebSocket server

We start `serve_websocket` as a background task in the notebook's event loop
and let it serve `index.html` on the same port. **Then open**
[http://localhost:8765/](http://localhost:8765/) in a browser before driving
the agent in the next cell.

In [ ]:
HOOK = WebSocketHook(thread_id_label="demo-live")
html_path = Path("index.html").resolve()

server_task = asyncio.create_task(
    serve_websocket(
        HOOK,
        host="localhost", port=8765, poll_interval=0.05,
        html_path=str(html_path),
    )
)
print("Open in a browser:  http://localhost:8765/")


## 4. Drive the agent — pass `HOOK` through `config["callbacks"]`

The handler subscribes to the `agent_pinboard:ingest` custom event that `@pin`
dispatches after each successful ingest. As deltas arrive on the WS, the
browser updates the Cytoscape graph in real time.

In [ ]:
llm = MockChatModel(plan=PLAN)
tools = [fetch_cloudtrail, vt_lookup, *make_graph_tools()]
agent = create_agent(llm, tools, store=InMemoryStore())

# Run sync invoke off the event loop so it doesn't block server polling.
await asyncio.to_thread(
    agent.invoke,
    {"messages": [{"role": "user", "content": "Investigate suspicious AssumeRole."}]},
    {"configurable": {"thread_id": "demo-live"}, "callbacks": [HOOK]},
)
print("agent finished. WS server stays up — run the next cell to stop it.")


## 5. Stop the server

In [ ]:
server_task.cancel()
import contextlib
with contextlib.suppress(asyncio.CancelledError):
    await server_task
